# Phân Tích Các Yếu Tố Ảnh Hưởng đến Tỷ Lệ Chấp Nhận Xe Điện tại Mỹ

**Giai đoạn:** 2019 – 2023 | **Phạm vi:** 50 bang

---
Notebook này gọi trực tiếp các module trong `src/` để tái tạo toàn bộ quy trình phân tích.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

# Đảm bảo import được src/ khi chạy từ thư mục notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from src.data_cleaning import load_and_clean, descriptive_stats
from src.diagnostics  import (
    plot_ev_trend, plot_top10_states, plot_charging_vs_ev,
    plot_correlation_matrix, compute_vif, plot_vif,
    pearson_correlation_test, plot_residuals, residual_stats
)
from src.regression import (
    run_ols, anova_table, ttest_coefficients, print_model_summary,
    X_VARS_FULL, X_VARS_REDUCED, X_VARS_REDUCED2, TARGET
)

## Phần 1: Làm sạch & chuẩn bị dữ liệu

In [ ]:
df = load_and_clean('data/EV_Data.csv')
print(df.head())

## Phần 2: Thống kê mô tả

In [ ]:
desc = descriptive_stats(df)
print('\nBảng 2.1: Thống kê mô tả')
display(desc)

## Phần 3: Đánh giá sơ bộ dữ liệu

In [ ]:
plot_ev_trend(df)

In [ ]:
plot_top10_states(df, year=2023)

In [ ]:
plot_charging_vs_ev(df)

## Phần 4: Phân tích định lượng

In [ ]:
corr_matrix = plot_correlation_matrix(df)

In [ ]:
# VIF – mô hình rút gọn (sau khi loại bachelor_attainment)
df_vif = df.copy()
df_vif['ln_income'] = np.log(df_vif['per_cap_income'])

vif_features_r1 = [
    'stations_per_100k_vehicles', 'ln_income',
    'gasoline_price_per_gallon', 'price_cents_per_kwh',
    'trucks_share', 'incentives'
]

vif_df = compute_vif(df_vif, vif_features_r1)
display(vif_df)
plot_vif(vif_df, 'Sau khi loại bachelor_attainment')

In [ ]:
pearson_df = pearson_correlation_test(df, X_VARS_FULL)
print('\nBảng 4.2: Kiểm định Pearson')
display(pearson_df)

## Phần 5: Xây dựng mô hình OLS

In [ ]:
# --- Mô hình đầy đủ ---
model_full = run_ols(X_VARS_FULL, df)
print_model_summary(model_full, 'Mô hình đầy đủ')

In [ ]:
# --- Mô hình rút gọn 1: bỏ bachelor_attainment ---
model_r1 = run_ols(X_VARS_REDUCED, df)
print_model_summary(model_r1, 'Mô hình rút gọn 1')

print('\nBảng ANOVA:')
display(anova_table(model_r1, df))

print('\nKiểm định t từng hệ số:')
display(ttest_coefficients(model_r1, df))

In [ ]:
# --- Mô hình rút gọn 2: bỏ thêm price_cents_per_kwh ---
model_r2 = run_ols(X_VARS_REDUCED2, df)
print_model_summary(model_r2, 'Mô hình rút gọn 2')

print('\nBảng ANOVA:')
display(anova_table(model_r2, df))

print('\nKiểm định t từng hệ số:')
display(ttest_coefficients(model_r2, df))

In [ ]:
print(f"So sánh R²:")
print(f"  Đầy đủ    : {model_full['r_squared']:.4f}")
print(f"  Rút gọn 1 : {model_r1['r_squared']:.4f}")
print(f"  Rút gọn 2 : {model_r2['r_squared']:.4f}")

## Phần 6: Phân tích phần dư

In [ ]:
residuals = df[TARGET].values - model_r1['y_hat']
plot_residuals(residuals, model_r1['y_hat'])

print('\nThống kê phần dư:')
display(residual_stats(residuals))